# 📊 Multi-Touch Attribution — Shapley Value Model

This notebook implements **Shapley-value-based multi-touch attribution** for retail marketing channels.

Shapley values (from cooperative game theory) distribute credit fairly across all marketing touchpoints based on their *marginal contribution* to conversions.

## Sections
1. Setup & Data Loading
2. Journey Construction
3. Conversion Rate per Channel Combination (Coalition)
4. Shapley Value Computation
5. Attribution Results & Visualisation
6. Business Insights

In [ ]:
import itertools
import warnings
from collections import defaultdict
from math import factorial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded ✓')

## 1. Load Data & Simulate Customer Journeys

In [ ]:
# Load cleaned orders
df = pd.read_csv('../data/sample_orders.csv', parse_dates=['order_date'])
df.head()

In [ ]:
# Simulate multi-touch journeys: each customer may appear in multiple channels
# In a real scenario, this comes from your clickstream / CRM data
np.random.seed(42)
CHANNELS = ['organic', 'paid_search', 'email', 'social', 'direct']

# Build synthetic journey data (customer -> [touchpoints] -> converted?)
customer_ids = df['customer_id'].unique()

journeys = []
for cid in customer_ids:
    n_touches = np.random.randint(1, 5)
    touchpoints = list(np.random.choice(CHANNELS, size=n_touches, replace=False))
    converted = 1 if cid in df['customer_id'].values else 0
    journeys.append({'customer_id': cid, 'touchpoints': touchpoints, 'converted': converted})

journeys_df = pd.DataFrame(journeys)
print(f'Journey records: {len(journeys_df)}')
journeys_df.head(10)

## 2. Coalition Conversion Rates

In [ ]:
# Map each journey to its channel coalition (frozenset) and compute conversion rate
coalition_stats = defaultdict(lambda: {'conversions': 0, 'total': 0})

for _, row in journeys_df.iterrows():
    coalition = frozenset(row['touchpoints'])
    coalition_stats[coalition]['total'] += 1
    coalition_stats[coalition]['conversions'] += row['converted']

# Conversion rate per coalition
coalition_rates = {
    c: s['conversions'] / s['total'] if s['total'] > 0 else 0
    for c, s in coalition_stats.items()
}

print(f'Total unique coalitions observed: {len(coalition_rates)}')

## 3. Shapley Value Computation

In [ ]:
def shapley_value(channel: str, all_channels: list, coalition_rates: dict) -> float:
    """
    Compute the Shapley value for a single channel.

    For each coalition S that does not contain `channel`,
    compute the marginal contribution of adding `channel` to S,
    weighted by coalition size probability.
    """
    n = len(all_channels)
    others = [c for c in all_channels if c != channel]
    phi = 0.0

    for size in range(len(others) + 1):
        for subset in itertools.combinations(others, size):
            coalition_without = frozenset(subset)
            coalition_with    = frozenset(subset) | {channel}

            v_without = coalition_rates.get(coalition_without, 0.0)
            v_with    = coalition_rates.get(coalition_with,    0.0)

            weight = (factorial(size) * factorial(n - size - 1)) / factorial(n)
            phi += weight * (v_with - v_without)

    return phi


# Compute Shapley values for all channels
shapley_values = {}
for ch in CHANNELS:
    shapley_values[ch] = shapley_value(ch, CHANNELS, coalition_rates)

# Normalise to sum to 1 (percentage attribution)
total = sum(shapley_values.values())
shapley_pct = {ch: v / total * 100 for ch, v in shapley_values.items()}

shapley_df = pd.DataFrame([
    {'channel': ch, 'shapley_value': v, 'attribution_pct': shapley_pct[ch]}
    for ch, v in shapley_values.items()
]).sort_values('shapley_value', ascending=False)

shapley_df

## 4. Attribution Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — Shapley values
axes[0].barh(
    shapley_df['channel'],
    shapley_df['shapley_value'],
    color=sns.color_palette('Blues_r', len(shapley_df))
)
axes[0].set_title('Shapley Values by Channel', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Shapley Value (Marginal Contribution to Conversion Rate)')
axes[0].invert_yaxis()

# Pie chart — Attribution %
axes[1].pie(
    shapley_df['attribution_pct'],
    labels=shapley_df['channel'],
    autopct='%1.1f%%',
    colors=sns.color_palette('Set2', len(shapley_df)),
    startangle=140
)
axes[1].set_title('Attribution Share by Channel (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../dashboards/screenshots/mta_shapley_attribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✓')

## 5. Business Insights

In [ ]:
print('\n===== SHAPLEY ATTRIBUTION SUMMARY =====')
for _, row in shapley_df.iterrows():
    print(f"  {row['channel']:<15} → {row['attribution_pct']:.1f}% credit")

top_channel = shapley_df.iloc[0]['channel']
top_pct     = shapley_df.iloc[0]['attribution_pct']
print(f'\n💡 Top contributor: {top_channel} ({top_pct:.1f}%)')
print('   → Prioritise budget allocation toward this channel.')
print('\n📌 Note: Shapley attribution is more equitable than last-click')
print('   or first-click models — it captures assist value across the funnel.')